# MolT5 Amine Representation: Enzyme x Amine Combination

Tests MolT5-derived amine embeddings (pre-computed on Colab) against multiple enzyme
representations, compared to F_physchem_onehot and Morgan 1024 baselines.

**Amine representations tested:**
- MolT5-small (512-dim learned embeddings)
- MolT5-base (768-dim learned embeddings)
- F_physchem_onehot (15 physicochemical + 26 one-hot = 41 dims) — previous best
- Morgan 1024-bit fingerprint — original baseline

**Enzyme representations tested:**
- full_protein (mean of all residues, 1024-dim)
- unique (conservation < 0.5, mean pool, 1024-dim)
- noncons_max (conservation < 0.5, max pool, 1024-dim)
- cons_plus_noncons (conserved + non-conserved mean, 2048-dim)

**Models:** XGBoost (regularized) + Random Forest, with feature importance analysis.

**Note:** 4 amines missing SMILES (serotonin, tyramine, glyglycine, unconjugated) are excluded.

## 1. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import h5py
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score, log_loss
)

import xgboost as xgb

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors

from Bio import SeqIO

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")

MOLT5_DIR = OUTPUT_DIR / "model_outputs" / "molt5_amine"
MOLT5_DIR.mkdir(exist_ok=True, parents=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

N_SPLITS = 10
SEEDS = [42, 123, 456, 789, 1011, 2022, 3033, 4044, 5055, 6066]

# Amines to exclude (no SMILES available)
EXCLUDE_AMINES = {'serotonin', 'tyramine', 'glyglycine', 'unconjugated'}

print("Setup complete.")
print(f"Output directory: {MOLT5_DIR}")

## 2. Load Data

In [ ]:
# --- Per-residue ProtT5 embeddings ---
h5_path = DATA_DIR / "Seqs_list_total_per_residue.h5"
per_residue_embeddings = {}
with h5py.File(h5_path, 'r') as f:
    for key in f.keys():
        uniprot_id = key.split('_')[-1]
        per_residue_embeddings[uniprot_id] = f[key][:]
print(f"Per-residue embeddings: {len(per_residue_embeddings)} enzymes")

# --- Full protein embeddings ---
h5_full = DATA_DIR / "Seqs_list_total.h5"
full_embeddings = {}
with h5py.File(h5_full, 'r') as f:
    for key in f.keys():
        uniprot_id = key.split('_')[-1]
        full_embeddings[uniprot_id] = f[key][:]
print(f"Full protein embeddings: {len(full_embeddings)} enzymes")

# --- Conservation scores ---
df_cons = pd.read_csv(OUTPUT_DIR / "conservation_scores.csv")
df_core = df_cons[df_cons['gap_fraction'] < 0.5].copy()

# --- Alignment mapping ---
alignment_to_seq = {}
for record in SeqIO.parse(OUTPUT_DIR / "bsh_aligned.fasta", 'fasta'):
    parts = record.id.split('_')
    uniprot_id = parts[-1] if len(parts) > 1 else record.id
    seq = str(record.seq)
    mapping = {}
    seq_pos = 0
    for aln_pos, char in enumerate(seq):
        if char != '-':
            mapping[aln_pos] = seq_pos
            seq_pos += 1
    alignment_to_seq[uniprot_id] = mapping
overlap = set(alignment_to_seq.keys()) & set(per_residue_embeddings.keys())

# --- Activity labels (exclude missing amines) ---
df_activity = pd.read_csv(OUTPUT_DIR / "enzyme_amine_activity.csv")
controls = ['CTRL1', 'CTRL2', 'CTRL3', 'CTRL4', 'CTRL5', 'CTRL6', 'CTRL7']
canonical = ['taurine', 'glycine']
df_activity = df_activity[~df_activity['Enzyme'].isin(controls)]
df_activity = df_activity[~df_activity['Amine'].isin(canonical)]
df_activity = df_activity[~df_activity['Amine'].isin(EXCLUDE_AMINES)]
df_agg = df_activity.groupby(['Enzyme', 'Amine']).agg(
    active=('active_approach2', 'any'),
    n_products=('ProductName', 'count'),
    n_active_products=('active_approach2', 'sum')
).reset_index()
print(f"Activity data (excl. missing amines): {df_agg.shape[0]} pairs")
print(f"  Amines: {df_agg['Amine'].nunique()}, Enzymes: {df_agg['Enzyme'].nunique()}")
print(f"  Active: {df_agg['active'].sum()} ({df_agg['active'].mean():.1%})")

# --- Amine SMILES ---
df_smiles = pd.read_excel(DATA_DIR / "bsh_reactants_SMILES_corrected.xlsx")
name_map = {
    '2,3-Diaminopropinoic Acid': '2,3_diaminopropionic acid',
    '2-aminophenol': '2_aminophenol',
    '3-methoxytyramine HCl': '3_methoxytyramine',
    '4-aminophenol': '4_aminophenol',
    'L-Alanine': 'alanine',
    'L-Arginine': 'arginine',
    'Asparagine': 'asparagine',
    'Cadaverine': 'cadaverine',
    'L-Citrulline': 'citrulline',
    'L-Cysteine': 'cysteine',
    'Dopamine HCl': 'dopamine',
    'gamma-Aminobutyric acid >99%': 'gaba',
    'L-Glutamine': 'glutamine',
    'Glycyl-L-Valine': 'glyglycine',
    'L-Histidine': 'histidine',
    'L-Lysine': 'lysine',
    'L-Methionine': 'methionine',
    'L-Ornithine monohydrochloride': 'ornithine',
    'L-Phenylalanine': 'phenylalanine',
    'DL-Proline': 'proline',
    'Putrescine': 'putrescine',
    'L-Serine': 'serine',
    'L-Threonine': 'threonine',
    'Tryptamine': 'tryptamine',
}
amine_mols = {}
for _, row in df_smiles.iterrows():
    name = row['Compound_Name']
    smiles = row['SMILES']
    norm_name = name_map.get(name, name.lower().replace(' ', '_').replace('-', '_'))
    if pd.isna(smiles):
        continue
    smiles_clean = smiles.split('.')[0]
    mol = Chem.MolFromSmiles(smiles_clean)
    if mol is not None:
        amine_mols[norm_name] = mol

amines_needed = df_agg['Amine'].unique()
print(f"Amines in filtered data: {len(amines_needed)}")

## 3. Compute Enzyme Representations

In [ ]:
def get_nonconserved_embedding(enzyme_id, conservation_threshold, pooling='mean'):
    if enzyme_id not in per_residue_embeddings or enzyme_id not in alignment_to_seq:
        return None
    embed = per_residue_embeddings[enzyme_id]
    aln_map = alignment_to_seq[enzyme_id]
    variable_aln_positions = df_core[
        df_core['conservation_score'] < conservation_threshold
    ]['alignment_position'].values
    seq_positions = []
    for aln_pos in variable_aln_positions:
        if aln_pos in aln_map:
            seq_pos = aln_map[aln_pos]
            if seq_pos < len(embed):
                seq_positions.append(seq_pos)
    if len(seq_positions) == 0:
        return None
    selected = embed[seq_positions]
    if pooling == 'mean':
        return selected.mean(axis=0)
    elif pooling == 'max':
        return selected.max(axis=0)
    return selected.mean(axis=0)

def get_conserved_embedding(enzyme_id, conservation_threshold=0.95):
    if enzyme_id not in per_residue_embeddings or enzyme_id not in alignment_to_seq:
        return None
    embed = per_residue_embeddings[enzyme_id]
    aln_map = alignment_to_seq[enzyme_id]
    conserved_aln_positions = df_core[
        df_core['conservation_score'] >= conservation_threshold
    ]['alignment_position'].values
    seq_positions = []
    for aln_pos in conserved_aln_positions:
        if aln_pos in aln_map:
            seq_pos = aln_map[aln_pos]
            if seq_pos < len(embed):
                seq_positions.append(seq_pos)
    if len(seq_positions) == 0:
        return None
    return embed[seq_positions].mean(axis=0)

# Build 4 enzyme representation dicts
enzyme_repr = {}

enzyme_repr['full_protein'] = {eid: emb for eid, emb in full_embeddings.items()}

d = {}
for eid in overlap:
    emb = get_nonconserved_embedding(eid, 0.5, 'mean')
    if emb is not None: d[eid] = emb
enzyme_repr['unique'] = d

d = {}
for eid in overlap:
    emb = get_nonconserved_embedding(eid, 0.5, 'max')
    if emb is not None: d[eid] = emb
enzyme_repr['noncons_max'] = d

d = {}
for eid in overlap:
    noncons = get_nonconserved_embedding(eid, 0.5, 'mean')
    cons = get_conserved_embedding(eid, 0.95)
    if noncons is not None and cons is not None:
        d[eid] = np.concatenate([cons, noncons])
enzyme_repr['cons_plus_noncons'] = d

print(f"{'Representation':<22} {'Enzymes':>8} {'Dims':>6}")
print("-" * 40)
for name, d in enzyme_repr.items():
    sample = list(d.values())[0]
    print(f"{name:<22} {len(d):>8} {sample.shape[0]:>6}")

## 4. Build All Amine Representations

In [ ]:
# --- A: Morgan 1024 ---
repr_morgan = {}
for name, mol in amine_mols.items():
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    repr_morgan[name] = np.array(fp, dtype=np.float32)

# --- B: F_physchem_onehot ---
def compute_physicochemical(mol):
    return np.array([
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol), Descriptors.NumAromaticRings(mol),
        Descriptors.NumAliphaticRings(mol), Descriptors.FractionCSP3(mol),
        Descriptors.HeavyAtomCount(mol), rdMolDescriptors.CalcNumAmideBonds(mol),
        Descriptors.NumValenceElectrons(mol), Descriptors.MaxPartialCharge(mol),
        Descriptors.MinPartialCharge(mol),
        Descriptors.BalabanJ(mol) if Descriptors.BalabanJ(mol) != 0 else 0.0,
    ], dtype=np.float32)

N_PHYSCHEM = 15
all_amines_sorted = sorted(amines_needed)
amine_to_idx = {a: i for i, a in enumerate(all_amines_sorted)}
n_amines_onehot = len(all_amines_sorted)

repr_physchem_onehot = {}
for name in amines_needed:
    phys = compute_physicochemical(amine_mols[name]) if name in amine_mols else np.zeros(N_PHYSCHEM, dtype=np.float32)
    onehot = np.zeros(n_amines_onehot, dtype=np.float32)
    if name in amine_to_idx:
        onehot[amine_to_idx[name]] = 1.0
    repr_physchem_onehot[name] = np.concatenate([phys, onehot])

# --- C & D: MolT5 embeddings (pre-computed) ---
molt5_small_path = DATA_DIR / "molt5_small_amine_embeddings.csv"
molt5_base_path = DATA_DIR / "molt5_base_amine_embeddings.csv"

repr_molt5_small = {}
repr_molt5_base = {}

if molt5_small_path.exists():
    df_m5s = pd.read_csv(molt5_small_path)
    for _, row in df_m5s.iterrows():
        name = row['amine']
        vec = row.drop('amine').values.astype(np.float32)
        repr_molt5_small[name] = vec
    print(f"MolT5-small: {len(repr_molt5_small)} amines, {len(vec)} dims")
else:
    print(f"WARNING: {molt5_small_path} not found! Run Colab notebook first.")

if molt5_base_path.exists():
    df_m5b = pd.read_csv(molt5_base_path)
    for _, row in df_m5b.iterrows():
        name = row['amine']
        vec = row.drop('amine').values.astype(np.float32)
        repr_molt5_base[name] = vec
    print(f"MolT5-base: {len(repr_molt5_base)} amines, {len(vec)} dims")
else:
    print(f"WARNING: {molt5_base_path} not found! Run Colab notebook first.")

# Collect all available amine representations
amine_reprs = {}
amine_reprs['morgan_1024'] = repr_morgan
amine_reprs['physchem_onehot'] = repr_physchem_onehot
if repr_molt5_small:
    amine_reprs['molt5_small'] = repr_molt5_small
if repr_molt5_base:
    amine_reprs['molt5_base'] = repr_molt5_base

print(f"\nAmine representations available: {list(amine_reprs.keys())}")
for name, d in amine_reprs.items():
    sample = list(d.values())[0]
    print(f"  {name}: {len(d)} amines, {len(sample)} dims")

## 5. Build Feature Matrices (all combinations)

In [ ]:
# Build feature matrices for every enzyme x amine representation combination
feature_matrices = {}

for enz_name, enz_dict in enzyme_repr.items():
    for ami_name, ami_dict in amine_reprs.items():
        combo_key = f"{enz_name}__{ami_name}"
        X_list, y_list = [], []
        enzymes, amines = [], []
        for _, row in df_agg.iterrows():
            enzyme, amine = row['Enzyme'], row['Amine']
            if enzyme not in enz_dict or amine not in ami_dict:
                continue
            features = np.concatenate([enz_dict[enzyme], ami_dict[amine]])
            X_list.append(features)
            y_list.append(int(row['active']))
            enzymes.append(enzyme)
            amines.append(amine)
        if len(X_list) == 0:
            print(f"  SKIPPED {combo_key}: no valid pairs")
            continue
        X = np.array(X_list, dtype=np.float32)
        y = np.array(y_list, dtype=np.int32)
        feature_matrices[combo_key] = (X, y, enzymes, amines)

print(f"\nTotal combinations: {len(feature_matrices)}")
for key, (X, y, _, _) in feature_matrices.items():
    print(f"  {key}: X={X.shape}, active={y.mean():.1%}")

## 6. Train XGBoost + Random Forest (10 splits each)

In [ ]:
def enzyme_holdout_split_seed(X, y, enzymes, amines, seed, test_size=0.2, val_size=0.2):
    enzymes_arr = np.array(enzymes)
    amines_arr = np.array(amines)
    unique_enzymes = np.unique(enzymes_arr)
    profiles = np.array([y[enzymes_arr == e].mean() for e in unique_enzymes])
    bins = pd.cut(profiles, bins=5, labels=False)
    train_val_enz, test_enz = train_test_split(
        unique_enzymes, test_size=test_size, random_state=seed, stratify=bins)
    tv_profiles = np.array([y[enzymes_arr == e].mean() for e in train_val_enz])
    tv_bins = pd.cut(tv_profiles, bins=5, labels=False)
    train_enz, val_enz = train_test_split(
        train_val_enz, test_size=val_size, random_state=seed, stratify=tv_bins)
    train_mask = np.isin(enzymes_arr, train_enz)
    val_mask = np.isin(enzymes_arr, val_enz)
    test_mask = np.isin(enzymes_arr, test_enz)
    return {
        'X_train': X[train_mask], 'y_train': y[train_mask],
        'X_val': X[val_mask], 'y_val': y[val_mask],
        'X_test': X[test_mask], 'y_test': y[test_mask],
        'train_enz': train_enz, 'val_enz': val_enz, 'test_enz': test_enz,
    }


def train_and_evaluate(split_data, model_type='XGBoost'):
    X_train, y_train = split_data['X_train'], split_data['y_train']
    X_val, y_val = split_data['X_val'], split_data['y_val']
    X_test, y_test = split_data['X_test'], split_data['y_test']
    n_neg = (y_train == 0).sum()
    n_pos = max((y_train == 1).sum(), 1)
    
    if model_type == 'XGBoost':
        model = xgb.XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.05,
            scale_pos_weight=n_neg / n_pos,
            reg_alpha=1.0, reg_lambda=5.0,
            subsample=0.7, colsample_bytree=0.7,
            min_child_weight=5,
            random_state=42, early_stopping_rounds=30,
            eval_metric='logloss', n_jobs=-1)
        model.fit(X_train, y_train,
                  eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False)
        evals = model.evals_result()
        train_logloss_curve = evals['validation_0']['logloss']
        val_logloss_curve = evals['validation_1']['logloss']
    elif model_type == 'RF':
        model = RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=15,
            min_samples_split=10, max_features='sqrt',
            class_weight={0: 1.0, 1: n_neg / n_pos},
            random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        train_logloss_curve = None
        val_logloss_curve = None
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_proba_train = model.predict_proba(X_train)[:, 1]
    y_proba_val = model.predict_proba(X_val)[:, 1]
    
    metrics = {
        'train_logloss': log_loss(y_train, y_proba_train),
        'val_logloss': log_loss(y_val, y_proba_val),
        'test_logloss': log_loss(y_test, y_proba),
        'logloss_gap': log_loss(y_val, y_proba_val) - log_loss(y_train, y_proba_train),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'accuracy': accuracy_score(y_test, y_pred),
        'train_acc': model.score(X_train, y_train),
    }
    
    return metrics, model, train_logloss_curve, val_logloss_curve

print("Training functions ready.")

In [ ]:
%%time

all_results = []
all_models = {}  # {combo_key: {model_type: [models]}}
all_logloss_curves = {}  # {combo_key: [(train, val), ...]}

combo_keys = list(feature_matrices.keys())

for combo_key in combo_keys:
    X, y, enzymes, amines = feature_matrices[combo_key]
    enz_name, ami_name = combo_key.split('__')
    all_models[combo_key] = {'XGBoost': [], 'RF': []}
    all_logloss_curves[combo_key] = []
    
    for model_type in ['XGBoost', 'RF']:
        for i, seed in enumerate(SEEDS):
            split = enzyme_holdout_split_seed(X, y, enzymes, amines, seed=seed)
            metrics, model, train_ll, val_ll = train_and_evaluate(split, model_type)
            metrics['combo'] = combo_key
            metrics['enzyme_repr'] = enz_name
            metrics['amine_repr'] = ami_name
            metrics['model_type'] = model_type
            metrics['split_idx'] = i
            metrics['split_seed'] = seed
            all_results.append(metrics)
            all_models[combo_key][model_type].append(model)
            if model_type == 'XGBoost':
                all_logloss_curves[combo_key].append((train_ll, val_ll))
    
    # Progress
    xgb_results = [r for r in all_results if r['combo'] == combo_key and r['model_type'] == 'XGBoost']
    rf_results = [r for r in all_results if r['combo'] == combo_key and r['model_type'] == 'RF']
    xgb_roc = np.mean([r['roc_auc'] for r in xgb_results])
    rf_roc = np.mean([r['roc_auc'] for r in rf_results])
    print(f"{combo_key}: XGB ROC={xgb_roc:.3f}, RF ROC={rf_roc:.3f}")

df_results = pd.DataFrame(all_results)
print(f"\nTotal experiments: {len(df_results)}")
print("Done!")

## 7. Performance Comparison

In [ ]:
# Summary table: mean +/- std per combination per model type
summary_rows = []
for combo_key in combo_keys:
    enz_name, ami_name = combo_key.split('__')
    for model_type in ['XGBoost', 'RF']:
        mask = (df_results['combo'] == combo_key) & (df_results['model_type'] == model_type)
        df_sub = df_results[mask]
        row = {
            'combo': combo_key,
            'enzyme_repr': enz_name,
            'amine_repr': ami_name,
            'model_type': model_type,
            'total_dims': feature_matrices[combo_key][0].shape[1],
            'roc_auc_mean': df_sub['roc_auc'].mean(),
            'roc_auc_std': df_sub['roc_auc'].std(),
            'pr_auc_mean': df_sub['pr_auc'].mean(),
            'pr_auc_std': df_sub['pr_auc'].std(),
            'f1_mean': df_sub['f1'].mean(),
            'f1_std': df_sub['f1'].std(),
            'logloss_gap_mean': df_sub['logloss_gap'].mean(),
            'logloss_gap_std': df_sub['logloss_gap'].std(),
        }
        summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)

for model_type in ['XGBoost', 'RF']:
    print(f"\n{'='*110}")
    print(f"{model_type} Results (ranked by PR-AUC):")
    print(f"{'='*110}")
    df_mt = df_summary[df_summary['model_type'] == model_type].sort_values('pr_auc_mean', ascending=False)
    print(f"{'Enzyme':<22} {'Amine':<18} {'Dims':>5} {'ROC-AUC':>14} {'PR-AUC':>14} {'F1':>12} {'LL Gap':>12}")
    print("-" * 110)
    for _, r in df_mt.iterrows():
        print(f"{r['enzyme_repr']:<22} {r['amine_repr']:<18} {r['total_dims']:>5.0f} "
              f"{r['roc_auc_mean']:.3f}+/-{r['roc_auc_std']:.3f} "
              f"{r['pr_auc_mean']:.3f}+/-{r['pr_auc_std']:.3f} "
              f"{r['f1_mean']:.3f}+/-{r['f1_std']:.3f} "
              f"{r['logloss_gap_mean']:>+.3f}+/-{r['logloss_gap_std']:.3f}")

In [ ]:
# Grouped bar chart: for each enzyme repr, compare amine reprs side by side
amine_repr_names = list(amine_reprs.keys())
enzyme_repr_names = list(enzyme_repr.keys())
amine_colors = {'morgan_1024': '#95a5a6', 'physchem_onehot': '#3498db',
                'molt5_small': '#e74c3c', 'molt5_base': '#2ecc71'}

fig, axes = plt.subplots(2, 2, figsize=(20, 14))

for ax, (metric, title) in zip(axes.flat,
    [('roc_auc', 'ROC-AUC'), ('pr_auc', 'PR-AUC'), ('f1', 'F1'), ('logloss_gap', 'Log Loss Gap')]):
    
    n_enz = len(enzyme_repr_names)
    n_ami = len(amine_repr_names)
    width = 0.8 / n_ami
    x = np.arange(n_enz)
    
    for j, ami_name in enumerate(amine_repr_names):
        means = []
        stds = []
        for enz_name in enzyme_repr_names:
            combo = f"{enz_name}__{ami_name}"
            row = df_summary[(df_summary['combo'] == combo) & (df_summary['model_type'] == 'XGBoost')]
            if len(row) > 0:
                means.append(row[f'{metric}_mean'].values[0])
                stds.append(row[f'{metric}_std'].values[0])
            else:
                means.append(0)
                stds.append(0)
        
        offset = (j - n_ami/2 + 0.5) * width
        ax.bar(x + offset, means, width, yerr=stds, label=ami_name,
               color=amine_colors.get(ami_name, 'gray'), alpha=0.8, capsize=3,
               edgecolor='black', linewidth=0.3)
    
    ax.set_xticks(x)
    ax.set_xticklabels(enzyme_repr_names, rotation=25, ha='right', fontsize=10)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(f'{title} (XGBoost)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    if metric == 'logloss_gap':
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8)

plt.suptitle('Amine Representation Comparison Across Enzyme Representations (XGBoost)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(MOLT5_DIR / 'amine_repr_comparison_xgb.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {MOLT5_DIR / 'amine_repr_comparison_xgb.png'}")

In [ ]:
# Same for RF
fig, axes = plt.subplots(2, 2, figsize=(20, 14))

for ax, (metric, title) in zip(axes.flat,
    [('roc_auc', 'ROC-AUC'), ('pr_auc', 'PR-AUC'), ('f1', 'F1'), ('logloss_gap', 'Log Loss Gap')]):
    
    n_enz = len(enzyme_repr_names)
    n_ami = len(amine_repr_names)
    width = 0.8 / n_ami
    x = np.arange(n_enz)
    
    for j, ami_name in enumerate(amine_repr_names):
        means = []
        stds = []
        for enz_name in enzyme_repr_names:
            combo = f"{enz_name}__{ami_name}"
            row = df_summary[(df_summary['combo'] == combo) & (df_summary['model_type'] == 'RF')]
            if len(row) > 0:
                means.append(row[f'{metric}_mean'].values[0])
                stds.append(row[f'{metric}_std'].values[0])
            else:
                means.append(0)
                stds.append(0)
        
        offset = (j - n_ami/2 + 0.5) * width
        ax.bar(x + offset, means, width, yerr=stds, label=ami_name,
               color=amine_colors.get(ami_name, 'gray'), alpha=0.8, capsize=3,
               edgecolor='black', linewidth=0.3)
    
    ax.set_xticks(x)
    ax.set_xticklabels(enzyme_repr_names, rotation=25, ha='right', fontsize=10)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(f'{title} (Random Forest)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    if metric == 'logloss_gap':
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8)

plt.suptitle('Amine Representation Comparison Across Enzyme Representations (Random Forest)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(MOLT5_DIR / 'amine_repr_comparison_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {MOLT5_DIR / 'amine_repr_comparison_rf.png'}")

## 8. Feature Importance Analysis

In [ ]:
# Enzyme vs amine block importance for each combination (XGBoost + RF)
importance_records = []

for combo_key in combo_keys:
    enz_name, ami_name = combo_key.split('__')
    enz_dims = list(enzyme_repr[enz_name].values())[0].shape[0]
    
    for model_type in ['XGBoost', 'RF']:
        models = all_models[combo_key][model_type]
        for i, model in enumerate(models):
            imp = model.feature_importances_
            enzyme_imp = imp[:enz_dims].sum()
            amine_imp = imp[enz_dims:].sum()
            total_imp = imp.sum()
            importance_records.append({
                'combo': combo_key,
                'enzyme_repr': enz_name,
                'amine_repr': ami_name,
                'model_type': model_type,
                'split_idx': i,
                'enzyme_frac': enzyme_imp / total_imp if total_imp > 0 else 0,
                'amine_frac': amine_imp / total_imp if total_imp > 0 else 0,
                'amine_feature_imp': imp[enz_dims:],
            })

df_imp = pd.DataFrame(importance_records)

# Summary table
for model_type in ['XGBoost', 'RF']:
    print(f"\n{'='*80}")
    print(f"Feature Importance Blocks ({model_type}):")
    print(f"{'='*80}")
    print(f"{'Enzyme repr':<22} {'Amine repr':<18} {'Enzyme %':>10} {'Amine %':>10}")
    print("-" * 65)
    for combo_key in combo_keys:
        enz_name, ami_name = combo_key.split('__')
        mask = (df_imp['combo'] == combo_key) & (df_imp['model_type'] == model_type)
        enz_frac = df_imp[mask]['enzyme_frac'].mean()
        ami_frac = df_imp[mask]['amine_frac'].mean()
        print(f"{enz_name:<22} {ami_name:<18} {enz_frac:>9.1%} {ami_frac:>9.1%}")

In [ ]:
# Amine importance fraction: grouped by amine representation
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, model_type in zip(axes, ['XGBoost', 'RF']):
    # For each amine repr, average amine_frac across enzyme reprs and splits
    ami_names_present = [a for a in amine_repr_names if any(a in k for k in combo_keys)]
    
    n_enz = len(enzyme_repr_names)
    n_ami = len(ami_names_present)
    width = 0.8 / n_ami
    x = np.arange(n_enz)
    
    for j, ami_name in enumerate(ami_names_present):
        fracs = []
        for enz_name in enzyme_repr_names:
            combo = f"{enz_name}__{ami_name}"
            mask = (df_imp['combo'] == combo) & (df_imp['model_type'] == model_type)
            if mask.sum() > 0:
                fracs.append(df_imp[mask]['amine_frac'].mean())
            else:
                fracs.append(0)
        
        offset = (j - n_ami/2 + 0.5) * width
        ax.bar(x + offset, fracs, width, label=ami_name,
               color=amine_colors.get(ami_name, 'gray'), alpha=0.8,
               edgecolor='black', linewidth=0.3)
    
    ax.set_xticks(x)
    ax.set_xticklabels(enzyme_repr_names, rotation=25, ha='right', fontsize=10)
    ax.set_ylabel('Amine Feature Importance Fraction', fontsize=11)
    ax.set_title(f'Amine Importance Fraction ({model_type})', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('How Much Importance Does Each Amine Representation Get?',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(MOLT5_DIR / 'amine_importance_fraction.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {MOLT5_DIR / 'amine_importance_fraction.png'}")

In [ ]:
# Log loss convergence for XGBoost: one panel per amine repr, overlay enzyme reprs
fig, axes = plt.subplots(1, len(amine_repr_names), figsize=(6 * len(amine_repr_names), 5))
if len(amine_repr_names) == 1:
    axes = [axes]

enz_colors = {'full_protein': '#3498db', 'unique': '#2ecc71',
              'noncons_max': '#e74c3c', 'cons_plus_noncons': '#9b59b6'}

for ax, ami_name in zip(axes, amine_repr_names):
    for enz_name in enzyme_repr_names:
        combo = f"{enz_name}__{ami_name}"
        if combo not in all_logloss_curves:
            continue
        curves = all_logloss_curves[combo]
        if not curves:
            continue
        min_len = min(len(c[0]) for c in curves)
        val_mean = np.mean([c[1][:min_len] for c in curves], axis=0)
        train_mean = np.mean([c[0][:min_len] for c in curves], axis=0)
        rounds = np.arange(min_len)
        color = enz_colors.get(enz_name, 'gray')
        ax.plot(rounds, train_mean, color=color, linewidth=1.5, linestyle='--', alpha=0.5)
        ax.plot(rounds, val_mean, color=color, linewidth=2, label=f'{enz_name}')
    
    ax.set_xlabel('Boosting Round', fontsize=10)
    ax.set_ylabel('Log Loss', fontsize=10)
    ax.set_title(f'{ami_name}\n(solid=val, dashed=train)', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Log Loss Convergence by Amine Representation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(MOLT5_DIR / 'logloss_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {MOLT5_DIR / 'logloss_convergence.png'}")

## 9. Save Results

In [ ]:
df_results.to_csv(MOLT5_DIR / 'molt5_combination_results.csv', index=False)
print(f"Saved: {MOLT5_DIR / 'molt5_combination_results.csv'} ({len(df_results)} rows)")

df_summary.to_csv(MOLT5_DIR / 'molt5_combination_summary.csv', index=False)
print(f"Saved: {MOLT5_DIR / 'molt5_combination_summary.csv'} ({len(df_summary)} rows)")

print(f"\nAll outputs saved to: {MOLT5_DIR}")

In [ ]:
# Final summary
print("=" * 100)
print("MOLT5 AMINE REPRESENTATION ANALYSIS - FINAL SUMMARY")
print("=" * 100)

for model_type in ['XGBoost', 'RF']:
    print(f"\n--- {model_type} (ranked by PR-AUC) ---")
    df_mt = df_summary[df_summary['model_type'] == model_type].sort_values('pr_auc_mean', ascending=False)
    print(f"{'Rank':>4} {'Enzyme':<22} {'Amine':<18} {'ROC-AUC':>14} {'PR-AUC':>14} {'LL Gap':>12}")
    print("-" * 90)
    for rank, (_, r) in enumerate(df_mt.iterrows(), 1):
        marker = ' ***' if rank == 1 else ''
        print(f"{rank:>4} {r['enzyme_repr']:<22} {r['amine_repr']:<18} "
              f"{r['roc_auc_mean']:.3f}+/-{r['roc_auc_std']:.3f} "
              f"{r['pr_auc_mean']:.3f}+/-{r['pr_auc_std']:.3f} "
              f"{r['logloss_gap_mean']:>+.3f}+/-{r['logloss_gap_std']:.3f}{marker}")

# Best overall
best = df_summary.sort_values('pr_auc_mean', ascending=False).iloc[0]
print(f"\n{'='*100}")
print(f"BEST OVERALL: {best['model_type']} + {best['enzyme_repr']} + {best['amine_repr']}")
print(f"  ROC-AUC: {best['roc_auc_mean']:.3f} +/- {best['roc_auc_std']:.3f}")
print(f"  PR-AUC:  {best['pr_auc_mean']:.3f} +/- {best['pr_auc_std']:.3f}")
print(f"  LL Gap:  {best['logloss_gap_mean']:+.3f}")
print("=" * 100)